# Part 3: NLP Pipeline & Sequence Modeling

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, Input


## Task 1: Dataset Understanding

In [ ]:
# Load the dataset

df = pd.read_csv('Data_Sets/part_3_nlp_sequence_modeling/customer_support_text_classification.csv')
print(f"Dataset shape: {df.shape}")
print(f"Number of records: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Dataset shape: (1500, 6)
Number of records: 1500

Columns: ['ticket_id', 'channel', 'customer_message', 'sentiment_label', 'word_count', 'urgent_flag']

Data types:
ticket_id             str
channel               str
customer_message      str
sentiment_label       str
word_count          int64
urgent_flag         int64
dtype: object

Missing values:
ticket_id           0
channel             0
customer_message    0
sentiment_label     0
word_count          0
urgent_flag         0
dtype: int64


In [2]:
# Sample text records

print("Sample text records:")
for label in df['sentiment_label'].unique():
    sample = df[df['sentiment_label'] == label]['customer_message'].iloc[0]
    print(f"\n[{label.upper()}]: {sample}")

Sample text records:

[NEUTRAL]: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.

[POSITIVE]: The refund process was fast and convenient. I appreciate the quick response.

[NEGATIVE]: My refund is still pending and this experience is frustrating. My ticket number is 33927.


In [3]:
# Target labels/classes and Class distribution

print("Target labels/classes:", df['sentiment_label'].unique())
print(f"Number of classes: {df['sentiment_label'].nunique()}")
print(f"\nClass distribution:")
print(df['sentiment_label'].value_counts())
print(f"\nClass distribution (%):")
print(df['sentiment_label'].value_counts(normalize=True).round(3) * 100)

Target labels/classes: <StringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str
Number of classes: 3

Class distribution:
sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64

Class distribution (%):
sentiment_label
neutral     34.9
negative    33.1
positive    31.9
Name: proportion, dtype: float64


**Observations:**
1. The dataset has 1500 records with 3 sentiment classes.
2. Classes are fairly balanced (neutral: ~35%, negative: ~33%, positive: ~32%).
3. There are no missing values in the dataset
4. Messages are relatively short customer support texts.

## Task 2: Text Preprocessing

In [ ]:
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

df['clean_text'] = df['customer_message'].apply(preprocess_text)

print("Before vs After preprocessing:")
for i in range(3):
    print(f"\nOriginal : {df['customer_message'].iloc[i]}")
    print(f"Cleaned  : {df['clean_text'].iloc[i]}")

Before vs After preprocessing:

Original : I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Cleaned  : need information payment process ticket number please respond soon possible

Original : I need information about the payment process.
Cleaned  : need information payment process

Original : The refund process was fast and convenient. I appreciate the quick response.
Cleaned  : refund process fast convenient appreciate quick response


**Preprocessing steps applied:**
1. **Lowercasing** — normalizes all text to lowercase
2. **Removing special characters** — keeps only alphabetic characters and spaces (removes ticket numbers, punctuation, etc.)
3. **Tokenization** — splits text into individual words using NLTK's word_tokenize
4. **Stopword removal** — removes common English words that don't carry much meaning (the, is, at, etc.)
5. **Short token removal** — removes single-character tokens

## Task 3: Text Vectorization
**Converting text into numerical format** using:
1. Bag of Words
2. TF-IDF

In [ ]:
# Splitting the data

X = df['clean_text']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 1200, Test size: 300


In [ ]:
# Method 1: Bag of Words (BoW)

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print(f"Method 1: Bag of Words (BoW)")
print("-----------------------------")
print(f"BoW - Train shape: {X_train_bow.shape}, Test shape: {X_test_bow.shape}")
print(f"BoW - Vocabulary size: {len(bow_vectorizer.vocabulary_)}")
print(f"BoW - Sample feature names: {list(bow_vectorizer.vocabulary_.keys())[:10]}")

# Method 2: Term Frequency-Inverse Document Frequency (TF-IDF)

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\nMethod 2: Term Frequency-Inverse Document Frequency (TF-IDF)")
print("-------------------------------------------------------------")
print(f"TF-IDF - Train shape: {X_train_tfidf.shape}, Test shape: {X_test_tfidf.shape}")
print(f"TF-IDF - Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"\nSample TF-IDF values for first document (non-zero entries):")
first_doc = X_train_tfidf[0].toarray().flatten()
non_zero_idx = first_doc.nonzero()[0]
feature_names = tfidf_vectorizer.get_feature_names_out()
for idx in non_zero_idx[:5]:
    print(f"  '{feature_names[idx]}': {first_doc[idx]:.4f}")


Method 1: Bag of Words (BoW)
-----------------------------
BoW - Train shape: (1200, 146), Test shape: (300, 146)
BoW - Vocabulary size: 146
BoW - Sample feature names: ['confirm', 'whether', 'ticket', 'assigned', 'please', 'respond', 'soon', 'possible', 'unhappy', 'account']

Method 2: Term Frequency-Inverse Document Frequency (TF-IDF)
-------------------------------------------------------------
TF-IDF - Train shape: (1200, 146), Test shape: (300, 146)
TF-IDF - Vocabulary size: 146

Sample TF-IDF values for first document (non-zero entries):
  'assigned': 0.4525
  'confirm': 0.4525
  'please': 0.2660
  'possible': 0.2982
  'respond': 0.2982


## Task 4: Baseline Model
**Building a simple baseline model** using:
1. Naive Bayes with Bag of Words
2. Logistic Regression with TF-IDF

Evaluating the models

In [ ]:
# Model 1: Naive Bayes Classifier with BoW

nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
nb_preds = nb_bow.predict(X_test_bow)

print(f"\nModel 1: Naive Bayes Classifier with BoW")
print("-----------------------------------------")

print(f"Accuracy: {accuracy_score(y_test, nb_preds):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, nb_preds))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, nb_preds))


Model 1: Naive Bayes Classifier with BoW
-----------------------------------------
Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        99
     neutral       1.00      1.00      1.00       105
    positive       1.00      1.00      1.00        96

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Confusion Matrix:
[[ 99   0   0]
 [  0 105   0]
 [  0   0  96]]


In [ ]:
# Model 2: Logistic Regression with TF-IDF

lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
lr_preds = lr_tfidf.predict(X_test_tfidf)

print(f"\nModel 2: Logistic Regression with TF-IDF")
print("-----------------------------------------")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_preds))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_preds))


Model 2: Logistic Regression with TF-IDF
-----------------------------------------
Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        99
     neutral       1.00      1.00      1.00       105
    positive       1.00      1.00      1.00        96

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Confusion Matrix:
[[ 99   0   0]
 [  0 105   0]
 [  0   0  96]]


**Inference:**
1. Both models - "Naive Bayes with Bag of Words" AND "Logistic Regression with TF-IDF" are outputing predictions with 100% accuracy.

## Task 5: Sequence Model or Conceptual Architecture
**Building a simple sequence model using LSTM**
I will build a bi-direction LSTM with the following architecture:
Input Layer (padded token ids, length=30) --> Embedding Layer (vocab=3000, dim=64) --> Bidirectional LSTM (64 units) --> Dropout (0.3) --> Dense Output Layer (3 units, softmax)

In [ ]:
# Encoding labels for the LSTM

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

y_train_cat = to_categorical(y_train_enc, num_classes=3)
y_test_cat = to_categorical(y_test_enc, num_classes=3)

print(f"Label mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")
print(f"y_train_cat shape: {y_train_cat.shape}")

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
y_train_cat shape: (1200, 3)


In [ ]:
EMBEDDING_DIM = 64
LSTM_UNITS = 64
MAX_VOCAB = 3000
MAX_LEN = 30

model = Sequential([
    Input(shape=(MAX_LEN,)), # Adding input layer with specified shape, since model summary was not showing output shape without it.
    Embedding(input_dim=MAX_VOCAB, output_dim=EMBEDDING_DIM),
    Bidirectional(LSTM(LSTM_UNITS)),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

# Compiling the model with Adam optimizer and categorical crossentropy loss
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 30, 64)         │       192,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 258,435 (1009.51 KB)

 Trainable params: 258,435 (1009.51 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
# Tokenizing and padding the text data for LSTM input
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

# Fitting the model on the training data

history = model.fit(
    X_train_pad, y_train_cat,
    epochs=15,
    batch_size=32,
    validation_split=0.15,
    verbose=1
)

Epoch 1/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.6275 - loss: 1.0081 - val_accuracy: 0.9611 - val_loss: 0.7713
Epoch 2/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9618 - loss: 0.3318 - val_accuracy: 1.0000 - val_loss: 0.0495
Epoch 3/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0207 - val_accuracy: 1.0000 - val_loss: 0.0041
Epoch 4/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0028 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 5/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0012 - val_accuracy: 1.0000 - val_loss: 6.7516e-04
Epoch 6/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 8.9252e-04 - val_accuracy: 1.0000 - val_loss: 5.0057e-04
Epoch 7/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 6.9551e-04 - val_accuracy: 1.0000 - val_loss: 3.9431e-04
Epoch 8/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 5.6582e-04 - 

In [24]:
# Evaluate LSTM on test set
lstm_loss, lstm_acc = model.evaluate(X_test_pad, y_test_cat, verbose=0)
lstm_preds_prob = model.predict(X_test_pad, verbose=0)
lstm_preds_idx = np.argmax(lstm_preds_prob, axis=1)
lstm_preds_labels = le.inverse_transform(lstm_preds_idx)

print("=" * 50)
print("Bidirectional LSTM")
print("=" * 50)
print(f"Test Accuracy: {lstm_acc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lstm_preds_labels))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, lstm_preds_labels))

Bidirectional LSTM
Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        99
     neutral       1.00      1.00      1.00       105
    positive       1.00      1.00      1.00        96

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Confusion Matrix:
[[ 99   0   0]
 [  0 105   0]
 [  0   0  96]]


In [25]:
# Final comparison of all models

print("Model Comparison Summary")
print("-----------------------------------------")
print(f"Logistic Regression + TF-IDF  : {accuracy_score(y_test, lr_preds):.4f}")
print(f"Naive Bayes + BoW             : {accuracy_score(y_test, nb_preds):.4f}")
print(f"Bidirectional LSTM            : {lstm_acc:.4f}")

Model Comparison Summary
-----------------------------------------
Logistic Regression + TF-IDF  : 1.0000
Naive Bayes + BoW             : 1.0000
Bidirectional LSTM            : 1.0000


**Summary**
1. Traditional (Logistic Regression + TF-IDF, Naive Bayes + BoW) came across as strong baselines for text classification, especially since the datasets were smaller. They are fast to train and performed well.
2. On the other hand, a Sequence models like LSTM can capture word order and context, which bag-of-words approaches lose. However, they need more data and training time to outperform simpler models on small datasets. In this case, it worked though and I got 100% accuracy with LSTM as well.

## Task 6: Attention and Transformer Reflection
Updating this in the README.md file.